## Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from libsvm.svmutil import *

## Dataloader

In [2]:
def load_data():
    train_X = []
    with open('data/X_train.csv', 'r') as file:
        for line in file:
            img = np.array(line.strip().split(','), dtype = np.float32)
            train_X.append(img)
        train_X = np.array(train_X)
    train_Y = []
    with open('data/Y_train.csv', 'r') as file:
        for line in file:
            label = int(line.strip())
            train_Y.append(label)
        train_Y = np.array(train_Y)
    test_X = []
    with open('data/X_test.csv', 'r') as file:
        for line in file:
            img = np.array(line.strip().split(','), dtype = np.float32)
            test_X.append(img)
        test_X = np.array(test_X)
    test_Y = []
    with open('data/Y_test.csv', 'r') as file:
        for line in file:
            label = int(line.strip())
            test_Y.append(label)
        test_Y = np.array(test_Y)
    return train_X, train_Y, test_X, test_Y

In [3]:
train_X, train_Y, test_X, test_Y = load_data()

## Build SVM model

In [4]:
class SVMModel():
    def __init__(self, params, mode = None):
        self.train_X, self.train_Y = train_X, train_Y
        self.test_X, self.test_Y = test_X, test_Y
        self.mode = mode
        if self.mode == 'custom':
            self.K = self.compute_train_K()
            self.model = svm_train(self.train_Y, self.K, params)
        else:
            self.prob = svm_problem(self.train_Y, self.train_X)
            self.param = svm_parameter(params)
            self.model = svm_train(self.prob, self.param)
    def predict(self):
        if self.mode == 'custom':
            self.K_star = self.compute_test_K()
            p_label, p_acc, p_val = svm_predict(self.test_Y, self.K_star, self.model)
        else:
            p_label, p_acc, p_val = svm_predict(self.test_Y, self.test_X, self.model)
    def grid_search(self):
        ####### for all kernel #######
        C_values = np.logspace(-2, 3, 6)
        ####### only for poly ########
        coef0_values = [0, 1, 2]
        degree_values = [2, 3, 4]
        ######for poly and RBF #######
        gamma_values = np.logspace(-2, 2, 5)
        kfolds = 5
        best_accuracy = 0
        best_params = dict()
        match self.mode:
            case 'linear':
                for C in C_values:
                    params = f'-s 0 -t 0 -c {C} -v {kfolds}'
                    accuracy = svm_train(self.prob, params)
                    if accuracy > best_accuracy:
                        best_accuracy = accuracy
                        best_params['C'] = C
                self.param = f"-s 0 -t 0 -c {best_params['C']}"
            case 'poly':
                for C in C_values:
                    for coef0 in coef0_values:
                        for degree in degree_values:
                            for gamma in gamma_values:
                                params = f'-s 0 -t 1 -c {C} -r {coef0} -d {degree} -g {gamma} -v {kfolds}'
                                accuracy = svm_train(self.prob, params)
                                if accuracy > best_accuracy:
                                    best_accuracy = accuracy
                                    best_params['C'] = C
                                    best_params['coef0'] = coef0
                                    best_params['degree'] = degree
                                    best_params['gamma'] = gamma
                self.param = f"-s 0 -t 1 -c {best_params['C']} -r {best_params['coef0']} -d {best_params['degree']} -g {best_params['gamma']}"
            case 'RBF':
                for C in C_values:
                    for gamma in gamma_values:
                        params = f'-s 0 -t 2 -c {C} -g {gamma} -v {kfolds}'
                        accuracy = svm_train(self.prob, params)
                        if accuracy > best_accuracy:
                            best_accuracy = accuracy
                            best_params['C'] = C
                            best_params['gamma'] = gamma
                self.param = f"-s 0 -t 2 -c {best_params['C']} -g {best_params['gamma']}"
        self.model = svm_train(self.prob, self.param)
    def linear_RBF_kernel(self, xa, xb, gamma = 1 / 784):
        linear = xa @ xb.T
        xa_square = np.sum(xa ** 2, axis=1).reshape(-1, 1)
        xb_square = np.sum(xb ** 2, axis=1).reshape(1, -1)
        l2_norm_matrix = xa_square + xb_square - 2 * (xa @ xb.T)
        RBF = np.exp(-gamma * l2_norm_matrix)
        return linear + RBF
    def compute_train_K(self):
        N = len(self.train_X)
        K = np.zeros((N, N + 1))
        K[:, 0] = np.arange(1, N + 1, dtype = np.int32)
        K[:, 1:] = self.linear_RBF_kernel(self.train_X, self.train_X)
        return K
    def compute_test_K(self):
        N = len(self.train_X)
        M = len(self.test_X)
        K_star = np.zeros((M, N + 1))
        K_star[:, 0] = np.arange(1, M + 1, dtype = np.int32)
        K_star[:,1:] = self.linear_RBF_kernel(self.test_X, self.train_X)
        return K_star

## Task 1: Use three different kernel functions

In [5]:
linear = SVMModel('-s 0 -t 0', mode = 'linear')
linear.predict()

*
optimization finished, #iter = 252
nu = 0.000448
obj = -0.448327, rho = 0.509188
nSV = 45, nBSV = 0
.*
optimization finished, #iter = 1857
nu = 0.004524
obj = -4.524783, rho = 1.006392
nSV = 119, nBSV = 0
*.*
optimization finished, #iter = 1179
nu = 0.002289
obj = -2.289537, rho = 0.384631
nSV = 92, nBSV = 0
*
optimization finished, #iter = 534
nu = 0.001391
obj = -1.391228, rho = 0.466255
nSV = 67, nBSV = 0
*.*
optimization finished, #iter = 1481
nu = 0.007807
obj = -8.029279, rho = 0.041919
nSV = 104, nBSV = 1
*.*
optimization finished, #iter = 1204
nu = 0.005193
obj = -5.193789, rho = 0.259796
nSV = 95, nBSV = 0
*
optimization finished, #iter = 694
nu = 0.002657
obj = -2.657532, rho = 0.053568
nSV = 65, nBSV = 0
....*..*
optimization finished, #iter = 6583
nu = 0.022888
obj = -22.971279, rho = -0.909500
nSV = 188, nBSV = 2
.*.*
optimization finished, #iter = 2370
nu = 0.006853
obj = -6.854273, rho = 0.543446
nSV = 139, nBSV = 0
*.*
optimization finished, #iter = 1140
nu = 0.003083

In [6]:
polynomial = SVMModel('-s 0 -t 1', mode = 'poly')
polynomial.predict()


*
optimization finished, #iter = 937
nu = 0.933628
obj = -1455.829068, rho = 0.888549
nSV = 1870, nBSV = 1866
*
optimization finished, #iter = 961
nu = 0.956140
obj = -1540.189711, rho = 0.666488
nSV = 1916, nBSV = 1910
*
optimization finished, #iter = 959
nu = 0.957168
obj = -1548.825170, rho = 0.689545
nSV = 1916, nBSV = 1913
*
optimization finished, #iter = 955
nu = 0.950890
obj = -1507.211995, rho = 0.853055
nSV = 1904, nBSV = 1899
*.*
optimization finished, #iter = 1000
nu = 1.000000
obj = -1815.710084, rho = -0.762777
nSV = 2000, nBSV = 2000
*.*
optimization finished, #iter = 1000
nu = 1.000000
obj = -1802.005796, rho = -0.742379
nSV = 2000, nBSV = 2000
*.*
optimization finished, #iter = 1000
nu = 1.000000
obj = -1869.957927, rho = -0.369399
nSV = 2000, nBSV = 2000
*.*
optimization finished, #iter = 1000
nu = 1.000000
obj = -1810.957687, rho = 0.020361
nSV = 2000, nBSV = 2000
*.*
optimization finished, #iter = 1000
nu = 1.000000
obj = -1816.478484, rho = 0.375705
nSV = 2000, nBSV

In [7]:
RBF = SVMModel('-s 0 -t 2', mode = 'RBF')
RBF.predict()

*
optimization finished, #iter = 132
nu = 0.058574
obj = -69.607515, rho = -0.714428
nSV = 132, nBSV = 104
*
optimization finished, #iter = 254
nu = 0.152807
obj = -208.549741, rho = 0.194710
nSV = 325, nBSV = 287
*
optimization finished, #iter = 261
nu = 0.143724
obj = -186.309254, rho = 0.082217
nSV = 307, nBSV = 266
*
optimization finished, #iter = 188
nu = 0.096988
obj = -120.423680, rho = -0.334820
nSV = 213, nBSV = 177
*
optimization finished, #iter = 259
nu = 0.174623
obj = -238.676959, rho = 1.653513
nSV = 365, nBSV = 331
*
optimization finished, #iter = 212
nu = 0.152967
obj = -212.126199, rho = 1.386022
nSV = 322, nBSV = 291
*
optimization finished, #iter = 163
nu = 0.111540
obj = -145.017498, rho = 0.236077
nSV = 240, nBSV = 211
*
optimization finished, #iter = 338
nu = 0.251592
obj = -365.998352, rho = -0.359891
nSV = 520, nBSV = 488
*
optimization finished, #iter = 309
nu = 0.184266
obj = -252.726755, rho = -0.628894
nSV = 390, nBSV = 349
*
optimization finished, #iter = 2

## Task 2: Grid Search

In [8]:
linear.grid_search()
linear.predict()

*
optimization finished, #iter = 140
nu = 0.025618
obj = -0.232524, rho = 0.362483
nSV = 61, nBSV = 22
*
optimization finished, #iter = 305
nu = 0.084648
obj = -0.919027, rho = 0.377867
nSV = 162, nBSV = 113
*
optimization finished, #iter = 378
nu = 0.070720
obj = -0.727876, rho = 0.334593
nSV = 145, nBSV = 91
*
optimization finished, #iter = 291
nu = 0.047218
obj = -0.467808, rho = 0.370407
nSV = 103, nBSV = 55
*
optimization finished, #iter = 246
nu = 0.106685
obj = -1.188029, rho = -0.127889
nSV = 192, nBSV = 151
*
optimization finished, #iter = 255
nu = 0.089983
obj = -1.017455, rho = -0.180044
nSV = 167, nBSV = 122
*
optimization finished, #iter = 167
nu = 0.060751
obj = -0.631999, rho = -0.051769
nSV = 112, nBSV = 82
*
optimization finished, #iter = 476
nu = 0.155909
obj = -1.823001, rho = -0.485782
nSV = 281, nBSV = 226
*
optimization finished, #iter = 350
nu = 0.110163
obj = -1.243009, rho = 0.277145
nSV = 204, nBSV = 150
*
optimization finished, #iter = 361
nu = 0.080186
obj =

In [9]:
polynomial.grid_search()
polynomial.predict()

*
optimization finished, #iter = 469
nu = 0.578462
obj = -6.201730, rho = 0.782481
nSV = 930, nBSV = 922
*
optimization finished, #iter = 560
nu = 0.681240
obj = -7.794564, rho = 0.575403
nSV = 1097, nBSV = 1081
*
optimization finished, #iter = 568
nu = 0.694514
obj = -7.943920, rho = 0.554762
nSV = 1115, nBSV = 1106
*
optimization finished, #iter = 538
nu = 0.645323
obj = -7.112673, rho = 0.756342
nSV = 1037, nBSV = 1029
*
optimization finished, #iter = 679
nu = 0.845801
obj = -10.069381, rho = -0.754659
nSV = 1357, nBSV = 1352
*
optimization finished, #iter = 669
nu = 0.832623
obj = -9.703612, rho = -0.739528
nSV = 1334, nBSV = 1330
*
optimization finished, #iter = 703
nu = 0.876375
obj = -10.137174, rho = -0.376552
nSV = 1404, nBSV = 1401
*
optimization finished, #iter = 701
nu = 0.864961
obj = -10.559937, rho = -0.033581
nSV = 1389, nBSV = 1377
*
optimization finished, #iter = 694
nu = 0.861885
obj = -10.309602, rho = 0.459551
nSV = 1383, nBSV = 1375
*
optimization finished, #iter 

In [10]:
RBF.grid_search()
RBF.predict()

*
optimization finished, #iter = 467
nu = 0.568952
obj = -5.464666, rho = -0.465862
nSV = 917, nBSV = 901
*
optimization finished, #iter = 701
nu = 0.865054
obj = -9.493676, rho = 0.091397
nSV = 1388, nBSV = 1380
*
optimization finished, #iter = 693
nu = 0.853098
obj = -9.349539, rho = -0.025789
nSV = 1369, nBSV = 1361
*
optimization finished, #iter = 637
nu = 0.781363
obj = -8.098828, rho = -0.075886
nSV = 1256, nBSV = 1246
*
optimization finished, #iter = 659
nu = 0.817306
obj = -8.871759, rho = 0.666641
nSV = 1311, nBSV = 1304
*
optimization finished, #iter = 646
nu = 0.803543
obj = -8.432839, rho = 0.558492
nSV = 1289, nBSV = 1283
*
optimization finished, #iter = 600
nu = 0.743689
obj = -7.629437, rho = 0.433066
nSV = 1194, nBSV = 1186
*
optimization finished, #iter = 777
nu = 0.967013
obj = -11.498280, rho = -0.153063
nSV = 1549, nBSV = 1546
*
optimization finished, #iter = 765
nu = 0.951486
obj = -10.771001, rho = -0.205754
nSV = 1524, nBSV = 1520
*
optimization finished, #iter =

In [13]:
print(linear.param)
print(polynomial.param)
print(RBF.param)

-s 0 -t 0 -c 0.1
-s 0 -t 1 -c 100.0 -r 2 -d 2 -g 10.0
-s 0 -t 2 -c 10.0 -g 0.01


## Task 3: Use linear kernel + RBF kernel (Custom kernel)

In [12]:
custom = SVMModel('-s 0 -t 4', mode = 'custom')
custom.predict()

*
optimization finished, #iter = 258
nu = 0.000447
obj = -0.447147, rho = 0.506032
nSV = 45, nBSV = 0
.*
optimization finished, #iter = 1672
nu = 0.004505
obj = -4.505073, rho = 1.001733
nSV = 119, nBSV = 0
*.*
optimization finished, #iter = 1196
nu = 0.002282
obj = -2.282267, rho = 0.387487
nSV = 92, nBSV = 0
*
optimization finished, #iter = 537
nu = 0.001387
obj = -1.386803, rho = 0.462077
nSV = 67, nBSV = 0
.*
optimization finished, #iter = 1564
nu = 0.007787
obj = -8.001534, rho = 0.042752
nSV = 104, nBSV = 1
*.*
optimization finished, #iter = 1183
nu = 0.005175
obj = -5.175967, rho = 0.261499
nSV = 95, nBSV = 0
*
optimization finished, #iter = 669
nu = 0.002650
obj = -2.650098, rho = 0.045397
nSV = 65, nBSV = 0
....*..*
optimization finished, #iter = 6642
nu = 0.022746
obj = -22.809472, rho = -0.899127
nSV = 188, nBSV = 1
.*.*
optimization finished, #iter = 2371
nu = 0.006827
obj = -6.827601, rho = 0.522922
nSV = 139, nBSV = 0
*.*
optimization finished, #iter = 1136
nu = 0.003074
